# Spike 03: gray_matter Frontmatter Extraction

**Goal:** Validate that `gray_matter` can:
1. Extract YAML frontmatter from standard markdown files
2. Return the body (markdown without frontmatter) separately
3. Handle files with no frontmatter
4. Handle edge cases: missing closing `---`, nested YAML, lists, booleans, dates

In [6]:
:dep gray_matter = "*"
:dep serde_yaml = "0.9"

## 1. Basic frontmatter extraction

In [7]:
use gray_matter::Matter;
use gray_matter::engine::YAML;
use gray_matter::ParsedEntity;

let matter = Matter::<YAML>::new();

let input = r#"---
title: "Simple Note"
tags: [rust, testing]
date: 2025-06-12
---

# Simple Note

Body content here.
"#;

let result: ParsedEntity<serde_yaml::Value> = matter.parse(input).unwrap();

println!("Has frontmatter data: {}", result.data.is_some());
println!("Body: {:?}", result.content.trim());

if let Some(data) = &result.data {
    println!("Frontmatter: {:#?}", data);
}

Has frontmatter data: true
Body: "# Simple Note\n\nBody content here."
Frontmatter: Mapping {
    "title": String("Simple Note"),
    "tags": Sequence [
        String("rust"),
        String("testing"),
    ],
    "date": String("2025-06-12"),
}


()

## 2. No frontmatter

In [8]:
let no_fm = "# Just a heading\n\nNo frontmatter here.";
let result: ParsedEntity<serde_yaml::Value> = matter.parse(no_fm).unwrap();

println!("Has data: {}", result.data.is_some());
println!("Body: {:?}", result.content.trim());

Has data: false
Body: "# Just a heading\n\nNo frontmatter here."


## 3. Extract as serde_yaml::Value for type inspection

In [9]:
let complex = r#"---
title: CRDT Notes
tags:
  - distributed-systems
  - crdt
date: 2025-03-20
draft: false
priority: 3
score: 0.95
nested:
  key: value
---

# Content
"#;

let result: ParsedEntity<serde_yaml::Value> = matter.parse(complex).unwrap();

if let Some(data) = &result.data {
    println!("Data type: {:?}", std::any::type_name_of_val(data));
    println!("Data: {:#?}", data);
}

Data type: "serde_yaml::value::Value"
Data: Mapping {
    "date": String("2025-03-20"),
    "title": String("CRDT Notes"),
    "score": Number(0.95),
    "draft": Bool(false),
    "priority": Number(3),
    "nested": Mapping {
        "key": String("value"),
    },
    "tags": Sequence [
        String("distributed-systems"),
        String("crdt"),
    ],
}


()

## 4. Edge cases

In [13]:
{
    // Edge case: missing closing ---
    let missing_close = "---\ntitle: Broken\n\n# Content without closing delimiter";
    match matter.parse::<serde_yaml::Value>(missing_close) {
        Ok(r) => println!("Missing close --- -> data: {:?}, body len: {}", r.data.is_some(), r.content.len()),
        Err(e) => println!("Missing close --- -> ERROR: {}", e),
    }

    // Edge case: empty frontmatter
    let empty_fm = "---\n---\n\n# Content";
    match matter.parse::<serde_yaml::Value>(empty_fm) {
        Ok(r) => println!("Empty frontmatter -> data: {:?}, body: {:?}", r.data.is_some(), r.content.trim()),
        Err(e) => println!("Empty frontmatter -> ERROR: {}", e),
    }

    // Edge case: frontmatter with only whitespace
    let ws_fm = "---\n  \n---\n\n# Content";
    match matter.parse::<serde_yaml::Value>(ws_fm) {
        Ok(r) => println!("Whitespace frontmatter -> data: {:?}", r.data.is_some()),
        Err(e) => println!("Whitespace frontmatter -> ERROR: {}", e),
    }

    // Edge case: triple-dash in content (not frontmatter)
    let dash_content = "# Title\n\nSome text\n\n---\n\nMore text after horizontal rule";
    match matter.parse::<serde_yaml::Value>(dash_content) {
        Ok(r) => println!("Dash in content -> data: {:?}, body len: {}", r.data.is_some(), r.content.len()),
        Err(e) => println!("Dash in content -> ERROR: {}", e),
    }
}

Missing close --- -> data: false, body len: 50
Empty frontmatter -> data: false, body: "# Content"
Whitespace frontmatter -> data: false
Dash in content -> data: false, body len: 56


()

## 5. Read from test fixtures

In [12]:
{
    let fixtures_dir = "../tests/fixtures";

    for entry in std::fs::read_dir(fixtures_dir).unwrap() {
        let entry = entry.unwrap();
        let path = entry.path();
        if path.extension().map_or(false, |e| e == "md") {
            let content = std::fs::read_to_string(&path).unwrap();
            match matter.parse::<serde_yaml::Value>(&content) {
                Ok(result) => println!(
                    "{}: frontmatter={}, body_len={}",
                    path.file_name().unwrap().to_string_lossy(),
                    result.data.is_some(),
                    result.content.len()
                ),
                Err(e) => println!(
                    "{}: ERROR: {}",
                    path.file_name().unwrap().to_string_lossy(),
                    e
                ),
            }
        }
    }
}

no-frontmatter.md: frontmatter=false, body_len=291
long-note.md: frontmatter=true, body_len=3738
rich-formatting.md: frontmatter=true, body_len=768
simple-note.md: frontmatter=true, body_len=503
short-note.md: frontmatter=true, body_len=46


()

## Findings

- [x] `parse` returns `Result<ParsedEntity<D>>` — needs type annotation, use `serde_yaml::Value`
- [x] Frontmatter data type: `serde_yaml::Value` — `Mapping`, `Sequence`, `String`, `Bool`, `Number`
- [x] `Number` doesn't distinguish int/float in Debug — use `.is_i64()` / `.is_f64()` on the inner `Number`
- [x] Dates come as `String("2025-03-20")` — type inference must parse date strings ourselves
- [x] No frontmatter → `data = None`, body is full content
- [x] Missing closing `---` → treated as no frontmatter (safe, no crash)
- [x] Empty / whitespace frontmatter → `data = None`
- [x] `---` horizontal rule in body → not mistaken for frontmatter
- [x] All test fixtures parse correctly